# Agent Roles Library - CrewAI Integration

This notebook demonstrates how to use the Agent Roles Library with **CrewAI**.

## What You'll Learn
- Load role definitions from YAML
- Apply CrewAI-specific overlays (goals, backstories)
- Create CrewAI agents from role definitions
- Build a multi-agent crew with specialized roles
- Execute tasks with the crew

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd().parent))

from crewai import LLM, Task, Crew
from loader import RoleRegistry
from models import RuntimeContext
from adapters.crewai_adapter import CrewAIAdapter

## Load API Key

In [ ]:
import os

# Set your API key via environment variable before running
# export OLLAMA_API_KEY="your-api-key-here"
api_key = os.getenv("OLLAMA_API_KEY", "your-api-key-here")

# Create LLM instance
llm = LLM(
    model="kimi-k2.7-code:cloud",
    base_url="https://ollama.com/v1",
    api_key=api_key,
)

## Step 1: Index the Role Registry

In [ ]:
registry = RoleRegistry()
registry.index()

print(f"Total roles loaded: {len(registry.list_roles())}")
print(f"Total overlays: {len(registry._overlays)}")

# List audit roles
audit_roles = [r for r in registry.list_roles() if r.category == "audit"]
print("\nAudit roles:")
for r in audit_roles:
    print(f"  - {r.name}")

## Step 2: Create a CrewAI Agent from a Role

In [ ]:
# Load a role with its CrewAI overlay
role, overlay = registry.get_role_with_overlay("lead_internal_auditor", "crewai")

print(f"Role: {role.name}")
print(f"Description: {role.description[:100]}...")
print(f"\nOverlay found: {overlay is not None}")
if overlay:
    print(f"Goal: {overlay.data.get('goal', 'N/A')}")
    print(f"Backstory: {overlay.data.get('backstory', 'N/A')[:100]}...")

In [ ]:
# Create runtime context
context = RuntimeContext(
    llm=llm,
    tools=[],
    allow_delegation=False,
)

# Adapt to CrewAI Agent
adapter = CrewAIAdapter(context)
agent = adapter.adapt(role, overlay.data if overlay else None)

print(f"Created CrewAI Agent: {agent.role}")
print(f"Goal: {agent.goal}")
print(f"Backstory: {agent.backstory[:150]}...")

## Step 3: Build a Multi-Agent Crew

In [ ]:
# Create multiple agents
roles_config = [
    ("lead_internal_auditor", "crewai"),
    ("it_auditor", "crewai"),
    ("compliance_auditor", "crewai"),
]

agents = []
for role_id, framework in roles_config:
    role, overlay = registry.get_role_with_overlay(role_id, framework)
    agent = adapter.adapt(role, overlay.data if overlay else None)
    agents.append(agent)
    print(f"Created: {agent.role}")

In [ ]:
# Define tasks
task1 = Task(
    description="Assess the organization's IT general controls and identify key risk areas in cloud infrastructure.",
    expected_output="A prioritized list of IT control gaps with risk ratings and remediation recommendations.",
    agent=agents[1],  # IT Auditor
)

task2 = Task(
    description="Evaluate compliance with GDPR and SOX requirements across data handling processes.",
    expected_output="A compliance gap analysis with remediation action plan.",
    agent=agents[2],  # Compliance Auditor
)

task3 = Task(
    description="Synthesize IT audit and compliance findings into an integrated risk report for the audit committee.",
    expected_output="An executive summary report with integrated risk ratings and board-level recommendations.",
    agent=agents[0],  # Lead Internal Auditor
)

In [ ]:
# Create and run crew
crew = Crew(
    agents=agents,
    tasks=[task1, task2, task3],
    verbose=True,
)

result = crew.kickoff()
print("\n" + "="*60)
print("CREW EXECUTION COMPLETE")
print("="*60)
print(result)

## Step 4: Dynamic Agent Creation from Search

In [ ]:
# Search for risk-related roles
risk_roles = registry.search("risk")
print(f"Found {len(risk_roles)} risk-related roles:")
for role in risk_roles:
    print(f"  - {role.name} ({role.category})")

In [ ]:
# Create agents dynamically from search results
risk_agents = []
for role in risk_roles[:2]:
    overlay = registry.get_overlay("crewai", role.id)
    agent = adapter.adapt(role, overlay.data if overlay else None)
    risk_agents.append(agent)
    print(f"Created risk agent: {agent.role}")

## Summary

This notebook demonstrated:
1. Loading role definitions from YAML files
2. Applying CrewAI overlays for goals and backstories
3. Creating agents using the `CrewAIAdapter`
4. Building multi-agent crews with specialized roles
5. Dynamic agent creation from registry searches

The Agent Roles Library decouples role definitions from framework implementations,
enabling you to maintain a single source of truth for agent personas while adapting
them to CrewAI's specific requirements.